# modernvbert

> ModernVBERT under the sysone head: its image tiles spliced into Laya's row, the pixel batches its encoder expects, and its image defaults, so the multimodal application runs it beside NeoMME.

In [ ]:
#| default_exp modernvbert

In [ ]:
#| export
import re

import numpy as np
import torch

from transformers import Idefics3Processor, ModernVBertConfig

from sysone.core import *
from sysone.template import *
from sysone.models import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import json, logging, tempfile
from pathlib import Path
from PIL import Image as PILImage
from transformers import AutoModel, AutoProcessor
from sysone.inference import Decider
from sysone.data import TypedDecisions
from sysone.multimodal import decision_learner
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

This notebook adds a second multimodal encoder the way the [multimodal](20_neomme.ipynb) notebook added NeoMME, and follows its outline: load and inspect, a row by hand, the template, the spec, one forward pass, a head-only run, and what to watch. Adding an encoder takes methods, not an application: ModernVBERT's rows, its inputs and its image defaults are methods of sysone's [generic functions](00_core.ipynb#generic-functions-and-multiple-dispatch), and the multimodal application runs it as it runs NeoMME.

[ModernVBERT](https://arxiv.org/abs/2510.01149) (Illuin Technology, 2025; MIT) is a bidirectional vision-language encoder built for visual document retrieval. It has 252M parameters. The text tower is a ModernBERT (22 layers, width 768, with global attention every third layer and a 128-token window in between), and the image tower is SigLIP2 (12 layers, 512-pixel tiles, 16-pixel patches). It fuses the two early. A tile's 32 × 32 patch vectors are pixel-shuffled into 64 vectors, each projected to the text's width, and they take the place of 64 `<image>` tokens in the text. The text tower then reads text and image as one sequence. Its pretraining was masked language modelling on image-text pairs, mostly documents, before the contrastive stage that made the retrievers.

For sysone this is the easy kind of multimodal encoder, for three reasons:

- **Its text tower is ModernBERT, Laya's own family.** The row uses Laya's layout, with `[CLS]`, `[SEP]` and a `[MASK]` anchor before each option, as for any ModernBERT.
- **An image takes as many positions in the output as it has tokens in the input.** So the anchors sit where the row put them, and gathering them works as it does for text. NeoMME needed two-axis position ids built by hand.
- **Its processor, transformers' `Idefics3Processor`, accepts text and images mixed in one prompt.** So sysone's image block can be checked token for token against it. NeoMME's processor refuses such rows.

```{mermaid}
flowchart LR
    R["a record: an image,<br/>a question, the gold"] --> T["the Image transform:<br/>load, resize to image_side"]
    T --> P["Idefics3's image processor:<br/>512-px tiles, plus the global<br/>image when there are several"]
    P --> W["a row: Laya's layout, the image<br/>as 64 <image> tokens per tile"]
    W --> E["ModernVBERT: SigLIP2 → pixel shuffle<br/>→ projection, into ModernBERT"]
    E --> A["a vector at<br/>each anchor"]
    A --> S["the head: standardise,<br/>score each anchor"]
```

## 1. Load and inspect

The fixture is a random ModernVBERT: a 2-layer, 64-wide ModernBERT beside a 2-layer, 32-wide SigLIP with 64-pixel tiles, whose 16 patches pixel-shuffle into 4 image tokens. Its processor is a real `Idefics3Processor` over the tiny ModernBERT tokenizer, with ModernVBERT's image tokens added. It loads by its config like any Hub checkpoint: `EncoderSpec.from_pretrained` sees the vision config, loads the processor, and the family `modernvbert` gives the template and the LoRA targets.

In [ ]:
FIX = "fixtures/tiny-modernvbert"
spec = EncoderSpec.from_pretrained(FIX)
proc, tok = spec.processor, spec.tokenizer
ip = proc.image_processor
spec, type(proc).__name__, {t: tok.convert_tokens_to_ids(t) for t in ("<image>", "<fake_token_around_image>", "<global-img>", "[MASK]")}

(EncoderSpec(fixtures/tiny-modernvbert: modernvbert, width 64, context 1024, multimodal, template modernvbert),
 'Idefics3Processor',
 {'<image>': 2049,
  '<fake_token_around_image>': 2048,
  '<global-img>': 2051,
  '[MASK]': 4})

In [ ]:
test_eq((spec.family, spec.modality, spec.template.name, spec.width), ("modernvbert", "multimodal", "modernvbert", 64))
test_eq((proc.image_seq_len, ip.max_image_size["longest_edge"], ip.do_image_splitting), (4, 64, True))

The processor resizes an image so that its long side is `size["longest_edge"]`, enlarging it if need be. It then cuts the result into tiles of `max_image_size` and adds the whole image, shrunk to one tile, as a global view. A single-tile image gets no global view: it is its own. Each tile becomes `image_seq_len` `<image>` tokens, after a `<fake_token_around_image>` and a tile tag:

In [ ]:
img = PILImage.new("RGB", (28, 28), (20, 20, 20))
out = proc(text=["<image>"], images=[[img]], add_special_tokens=False, return_tensors="pt")
print(tok.decode(out["input_ids"][0]).replace("<image>" * 4, "<image>×4 "))
{k: tuple(v.shape) for k, v in out.items()}

<fake_token_around_image><row_1_col_1><image>×4 <fake_token_around_image><row_1_col_2><image>×4 
<fake_token_around_image><row_2_col_1><image>×4 <fake_token_around_image><row_2_col_2><image>×4 

<fake_token_around_image><global-img><image>×4 <fake_token_around_image>


{'input_ids': (1, 34),
 'attention_mask': (1, 34),
 'pixel_values': (1, 5, 3, 64, 64),
 'pixel_attention_mask': (1, 5, 64, 64)}

The real checkpoint's processor enlarges every image to 2,048 pixels on its long side. A 28-pixel MNIST digit therefore becomes 16 tiles of 512 pixels plus the global view, 17 × 64 = 1,088 image tokens, where a single 512-pixel tile would read the same strokes with 64. Measured with the checkpoint's processor: So the multimodal application sets `size["longest_edge"]` from its `image_side` (one 512-pixel tile by default), and a small image becomes one tile:

| `image_side` | tiles of a square image | the image block's tokens |
| --- | --- | --- |
| up to 512 | 1 | 67: 64 `<image>`, 3 markers |
| 1,024 | 2 × 2 and the global view | 333: 320 `<image>`, 13 markers and newlines |
| 1,536 | 3 × 3 and the global view | 664: 640 `<image>`, 24 others |
| 2,048 (the checkpoint's default) | 4 × 4 and the global view | 1,127: 1,088 `<image>`, 39 others |

A 3:2 photo gets as many tiles as a square up to 1,536 pixels, because the processor rounds both sides up to whole tiles, and 4 × 3 and the global view at 2,048.

Below 512 the image gets no fewer tokens, only less detail: the processor enlarges it back to the tile's 512 pixels.

## 2. A row by hand

A decision row puts the image between the options and the state, as the `modernvbert` preset lays it out. The image block is exactly what the processor makes of an `<image>` in a prompt: the tiles' tokens, row by row, then the global view. `idefics3_block` asks the image processor for the tiles and the processor for their text, and tokenizes that text without special tokens, as the rest of the row is tokenized:

In [ ]:
#| export
IDEFICS3_TOKENS = ("<fake_token_around_image>", "<global-img>") + tuple(f"<row_{i}_col_{j}>" for i in range(1, 7) for j in range(1, 7))

def idefics3_block(builder:RowBuilder, image) -> tuple:
    "The ids of one image as an Idefics3 processor lays it out (its tiles, then the global view), and its pixels and pixel mask"
    proc = builder.processor
    feat = proc.image_processor(images=[[image]], return_row_col_info=True, return_tensors="pt")
    text = proc.replace_image_token({"rows": feat["rows"], "cols": feat["cols"]}, 0)
    return builder.tok(text, add_special_tokens=False)["input_ids"], feat["pixel_values"][0], feat["pixel_attention_mask"][0]

def build_idefics3_rows(processor:Idefics3Processor, builder:RowBuilder, record:dict, *, skip_errors:bool=False) -> list:
    "Rows of a transformed record for an Idefics3-style processor: text by the builder's budgets, the image as tiles of <image> tokens"
    images = record.get("images") or []
    if len(images) > 1: raise ValueError("a ModernVBERT row holds one image")
    for s in IDEFICS3_TOKENS:                                     # user text never smuggles in an image token
        if s not in builder.extra_scrub: builder.extra_scrub.append(s)
    ids_img, pixels, mask = idefics3_block(builder, images[0]) if images else (None, None, None)
    extras = {"pixel_values": pixels, "pixel_attention_mask": mask, "image_tiles": len(pixels)} if images else {"image_tiles": 0}
    gold, rows = record.get("gold") or {}, []
    for qid, q, ids, markers, _, spans in builder._layout(record, ids_img, skip_errors):
        target, label = q.target(gold.get(qid))
        rows.append(Row(ids, markers, q.qtype, target, label, meta={"qid": qid, "case": record.get("id")}, extras=dict(extras),
                        spans=spans, answerable=q.answerable(gold.get(qid))))
    return rows

processor_rows.register(build_idefics3_rows)
EXTRA_COLLATE["pixel_attention_mask"] = "cat"            # tiles concatenate, as `pixel_values` do; the hook below regroups them

In [ ]:
qs = {"digit": {"type": "choice", "instructions": "Which digit is written in the image?", "criteria": [str(d) for d in range(4)]}}
rec = {"id": "r0", "state": {"text": "", "image": img}, "questions": qs, "gold": {"digit": "3"}}
row = spec.builder(tfms=[Image(side=None)]).build_record(rec)[0]
print(show_row(row, tok, max_chars=400))
blk, pixels, pmask = idefics3_block(spec.builder(), img)
test_eq(blk, out["input_ids"][0].tolist())                              # the processor's own layout of the image
test_eq((torch.equal(pixels, out["pixel_values"][0]), torch.equal(pmask, out["pixel_attention_mask"][0])), (True, True))
at = row.markers[-1] + 3                                                    # past the last option and the [SEP] after it
test_eq(row.ids[at:at + len(blk)], blk)
test_eq((row.extras["image_tiles"], row.extras["pixel_values"].shape), (5, (5, 3, 64, 64)))

[CLS] choice question: Which digit is written in the image? [SEP] [MASK] 0 [MASK] 1 [MASK] 2 [MASK] 3 [SEP] <fake_token_around_image> <row_1_col_1> <image>×4 <fake_token_around_image> <row_1_col_2> <image>×4 <fake_token_around_image> <row_2_col_1> <image>×4 <fake_token_around_image> <row_2_col_2> <image>×4 <fake_token_around_image> <global-img> <image>×4 <fake_token_around_image> [SEP]
markers [25, 27, 29, 31] · qtype choice · 69 tokens


## 3. The template

The `modernvbert` preset (in [template](01_template.ipynb#presets)) is Laya's row with an `{image}` field between the options and the state: `[CLS]`, the question, `[SEP]`, the anchored options, `[SEP]`, the image block, the state's text, `[SEP]`. Its budget is 2,048 tokens against Laya's 512, since an image at 2,048 pixels takes 1,127 positions. ModernBERT's 128-token local window is why the image sits right after the options: the first anchors are within a window of the image's first tiles, and every third layer attends globally anyway.

A user's text is scrubbed of the processor's image tokens, as NeoMME's `<doc>` and `<row>` are, so text can't open a fake image:

In [ ]:
b = spec.builder(tfms=[Image(side=None)])
r2 = b.build_record({"state": {"text": "a note <global-img> with <row_1_col_1> tags", "image": img}, "questions": qs})[0]
test_eq(r2.ids.count(tok.convert_tokens_to_ids("<global-img>")), 1)          # the image's own, not the text's
test_eq(r2.ids.count(tok.convert_tokens_to_ids("<row_1_col_1>")), 1)

## 4. The spec

The family `modernvbert` names the template and the LoRA targets: the text tower's attention projections, `Wqkv` and `Wo` in each of ModernBERT's layers. The image tower and the connector stay frozen under LoRA, as the retrievers built on ModernVBERT keep them. The multimodal application sets the processor's `size["longest_edge"]` to `image_side`, through ModernVBERT's method of `prepare_processor`, and adds the `Image` transform that loads each image and resizes it to the same side. An export saves the processor with its size, so a loaded `Decider` tiles images exactly as training did.

In [ ]:
test_eq(spec.lora_targets, r"text_model\.layers\.\d+\.attn\.(Wqkv|Wo)")

## 5. One forward pass

sysone's collate concatenates each row's tiles, as it does NeoMME's patches. ModernVBERT wants them grouped, `(batch, tiles, 3, H, W)`, with a pixel mask of the same shape and all-zero tiles as padding (its vision tower drops them). ModernVBERT's method of `encoder_inputs` regroups a batch's tiles by each row's `image_tiles`:

In [ ]:
#| export
def modernvbert_inputs(config:ModernVBertConfig, inputs:dict) -> dict:
    "A batch's concatenated tiles regrouped per row, (batch, max tiles, 3, H, W), zero-padded, and the pixel mask alike"
    kw = dict(inputs); n = kw.pop("image_tiles", None)
    pv, pm = kw.get("pixel_values"), kw.get("pixel_attention_mask")
    if n is None or pv is None: return kw
    n = [int(v) for v in n.reshape(-1).tolist()]
    if max(n) == 0: kw.pop("pixel_values"); kw.pop("pixel_attention_mask", None); return kw
    out = pv.new_zeros(len(n), max(n), *pv.shape[1:])
    mask = pm.new_zeros(len(n), max(n), *pm.shape[1:]) if pm is not None else None
    s = 0
    for b, k in enumerate(n):
        out[b, :k] = pv[s:s + k]
        if mask is not None: mask[b, :k] = pm[s:s + k]
        s += k
    kw["pixel_values"] = out
    if mask is not None: kw["pixel_attention_mask"] = mask
    return kw

encoder_inputs.register(modernvbert_inputs)

A very wide image makes fewer tiles than a square one: at the fixture's 128 pixels, a 400 × 50 banner is one row of two tiles plus the global view. Batched with the square digit's five tiles, the banner's row is padded with two empty tiles. Both rows' anchors then match what the model gives each row on its own, with the processor's layout:

In [ ]:
wide = PILImage.new("RGB", (400, 50), (200, 30, 30))
rows = [b.build_record(rec)[0], b.build_record(dict(rec, id="r1", state={"text": "a banner", "image": wide}))[0]]
test_eq([r.extras["image_tiles"] for r in rows], [5, 3])
enc = spec.load_encoder().eval()
model = EncoderDecisionModel(enc, DecisionHead(spec.width, layers=0, scorer="mlp"), spec).eval()
bt = collate(rows, tok.pad_token_id)
extra = {k: v for k, v in bt.items() if k in ("pixel_values", "pixel_attention_mask", "image_tiles")}
with torch.no_grad():
    h = model.encode(bt["input_ids"], bt["attention_mask"], **extra)
    for i, r in enumerate(rows):
        ids = torch.tensor([r.ids])
        alone = enc(input_ids=ids, attention_mask=torch.ones_like(ids), pixel_values=r.extras["pixel_values"][None],
                    pixel_attention_mask=r.extras["pixel_attention_mask"][None]).last_hidden_state[0]
        test_close(h[i, r.markers], alone[r.markers], eps=1e-5)
    logits = model(**{k: v for k, v in bt.items() if k not in ("target", "label")})
test_eq(logits.shape, (2, 4))
logits

tensor([[-0.1916, -0.1916, -0.1916, -0.1916],
        [-0.1917, -0.1917, -0.1917, -0.1917]])

## 6. A head-only run

The multimodal application, `sysone.multimodal.decision_learner`, runs ModernVBERT by default since 2026-09-30 (the [NeoMME notebook](20_neomme.ipynb#which-encoder-by-default) says why), so `decision_learner(data)` is `decision_learner(data, "ModernVBERT/modernvbert")`. The two methods below give it ModernVBERT's image defaults. `default_image_side` is one tile, the processor's own tile size (512 pixels for the checkpoint), and `prepare_processor` sets the processor to tile at the chosen side and counts the tiles. Everything else is the application's, as for NeoMME: the `Image` transform, a frozen encoder whose anchors are cached for the default head (in float32), and Laya's scorer on the standardised anchors. The same defaults make the two encoders comparable, in the [comparison](tutorials/neomme_vs_modernvbert.ipynb) notebook.

A batch of rows is a batch of tiles for the vision tower, so an `image_side` above 512 multiplies SigLIP's batch: 8 rows at 1,536 pixels are 80 tiles of 1,024 patches each. The application therefore divides the preset's batch sizes by the tiles per image, unless they are given, so the vision tower's batch stays about the size it has at one tile.

::: {.callout-note title="Finding: eight rows at 1,536 pixels held 8 GB of GPU memory"}
The first run of the [ModernVBERT tutorial](tutorials/modernvbert_images.ipynb)'s image-size sweep cached the anchors in batches of 8 rows, as the laptop preset says. At 1,536 pixels that is 80 tiles in one SigLIP pass, and 10 GB of the M3 Pro's 18 GB was wired: GPU memory, which macOS can't page out. Free memory fell to 16%, with swap all but full, and the run was stopped before it could take the machine down. Stopping it released 8.4 GB at once. Batches of one row at 1,536 pixels, 10 tiles, keep the vision tower's batch close to what 8 one-tile rows make.
:::

In [ ]:
#| export
ENCODER = "ModernVBERT/modernvbert"

def tiles_for(side:int|None, tile:int=512) -> int:
    "The tiles a square image makes at `side` pixels (None: the checkpoint's 2,048): one up to a tile, else a grid and the global view"
    n = -(-(side or 2048) // tile)
    return 1 if n <= 1 else n * n + 1

def idefics3_image_side(processor:Idefics3Processor) -> int:
    "An Idefics3 processor's default image side: one tile, the processor's own tile size (512 pixels for ModernVBERT)"
    return processor.image_processor.max_image_size["longest_edge"]

def idefics3_prepare(processor:Idefics3Processor, side:int|None) -> int:
    "Tile at `side` pixels (None: the processor's own size): the tiles a square image then makes, the global view included"
    if side is not None: processor.image_processor.size = {"longest_edge": side}
    return tiles_for(side, processor.image_processor.max_image_size["longest_edge"])

default_image_side.register(idefics3_image_side)
prepare_processor.register(idefics3_prepare)

Going back, from a row to the picture, ModernVBERT's method of [`pixel_tiles`](01_template.ipynb) stitches the tiles back into the image they were cut from, reading the grid off the row's `<row_i_col_j>` tokens, and returns the global view beside it. An image of one tile is that tile.

In [ ]:
#| export
def idefics3_tiles(processor:Idefics3Processor, row:Row, values=None) -> tuple:
    "ModernVBERT's pixels as the image they show: the tiles stitched back into the image (from its `<row_i_col_j>` tokens), then the global view"
    tiles, vals = tile_arrays(processor, row, values)
    grid = [tuple(map(int, m.groups())) for t in processor.tokenizer.convert_ids_to_tokens(list(row.ids)) if (m := re.fullmatch(r"<row_(\d+)_col_(\d+)>", t))]
    if len(tiles) < 2 or not grid: return tiles, vals
    R, C = max(r for r, _ in grid), max(c for _, c in grid)
    if R * C != len(tiles) - 1: return tiles, vals
    stitch = lambda xs, h, w: np.concatenate([np.concatenate(xs[r * C:(r + 1) * C], axis=w) for r in range(R)], axis=h)
    return [stitch(tiles[:-1], 0, 1), tiles[-1]], (None if vals is None else [stitch(vals[:-1], -2, -1), vals[-1]])

pixel_tiles.register(idefics3_tiles)

With NeoMME's adapter imported as well, the multimodal application finds a method for each encoder, and the fallback (`object`) for any other:

In [ ]:
{f.__name__: methods(f) for f in (processor_rows, default_image_side, prepare_processor, encoder_inputs, pixel_tiles)}

{'processor_rows': [('object', 'Any', 'dict'),
  ('NeoMMEProcessor', 'RowBuilder', 'dict'),
  ('Idefics3Processor', 'RowBuilder', 'dict')],
 'default_image_side': [('object',),
  ('NeoMMEProcessor',),
  ('Idefics3Processor',)],
 'prepare_processor': [('object', 'int | None'),
  ('Idefics3Processor', 'int | None')],
 'encoder_inputs': [('object', 'dict'),
  ('NeoMMEConfig', 'dict'),
  ('ModernVBertConfig', 'dict')],
 'pixel_tiles': [('object', 'Row', 'Any'),
  ('object', 'Row'),
  ('NeoMMEProcessor', 'Row', 'Any'),
  ('NeoMMEProcessor', 'Row'),
  ('Idefics3Processor', 'Row', 'Any'),
  ('Idefics3Processor', 'Row')]}

On the fixture, 24 images in three colours, and a choice among the colours. The encoder is random, so what matters is the path, from records to an exported `Decider` that answers as the learner did:

In [ ]:
tmp = Path(tempfile.mkdtemp())
COLOURS = {"red": (200, 30, 30), "green": (30, 170, 60), "blue": (40, 60, 200)}
cq = {"colour": {"type": "choice", "instructions": "Which colour fills the image?", "criteria": list(COLOURS)}}
crecs = []
for i in range(24):
    name = list(COLOURS)[i % 3]
    p = tmp / f"{i}.png"; PILImage.new("RGB", (40 + 7 * i, 40), COLOURS[name]).save(p)
    crecs.append({"id": f"c{i}", "state": {"text": "", "image": str(p)}, "questions": cq, "gold": {"colour": name}})
cdata = TypedDecisions.from_records(crecs, valid=0.25, calib=0.25, seed=0)
torch.manual_seed(0)
learn = decision_learner(cdata, FIX, image_side=64, preset="cpu", loss="soft_ce", cache_dir=tempfile.mkdtemp(), calibrate_after_fit=False)
test_eq((learn.cache, learn.model.head.scorer_kind, learn.model.head.layers), ("masks", "laya", 0))      # Laya's scorer on the anchors
test_eq((default_image_side(proc), learn.data.builder.template.name), (64, "modernvbert"))           # one of the fixture's 64-pixel tiles
test_eq(cdata.builder.processor.image_processor.size, {"longest_edge": 64})
learn.fit(3, lr=3e-3)
test_eq(learn.dataset("train")[0]["anchors"].shape, (3, 64))
r = crecs[1]
before = learn.predict(r["state"], r["questions"])
out = learn.export(tmp / "colour-jev")
jev = Decider.load(out, device="cpu")
test_eq(jev.predict(r["state"], r["questions"]), before)
test_eq(jev.builder.processor.image_processor.size["longest_edge"], 64)       # the export tiles as training did
before

{'eval_loss': '1.096', 'eval_accuracy': '0.3333', 'eval_soft_accuracy': '0.3341', 'eval_brier': '0.6652', 'eval_kl': '1.096', 'eval_tv': '0.6659', 'eval_ece': '0.001701', 'eval_score_mae': 'nan', 'eval_within_one': 'nan', 'eval_n': 6, 'eval_accuracy_choice': '0.3333', 'eval_accuracy_score': 'nan', 'eval_accuracy_noul': 'nan', 'eval_n_unanswerable': 0, 'eval_runtime': '0.002', 'eval_samples_per_second': '3072', 'eval_steps_per_second': '512', 'epoch': '1'}
{'eval_loss': '1.092', 'eval_accuracy': '0.5', 'eval_soft_accuracy': '0.3356', 'eval_brier': '0.6621', 'eval_kl': '1.092', 'eval_tv': '0.6644', 'eval_ece': '0.1638', 'eval_score_mae': 'nan', 'eval_within_one': 'nan', 'eval_n': 6, 'eval_accuracy_choice': '0.5', 'eval_accuracy_score': 'nan', 'eval_accuracy_noul': 'nan', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0017', 'eval_samples_per_second': '3591', 'eval_steps_per_second': '598.4', 'epoch': '2'}
{'eval_loss': '1.087', 'eval_accuracy': '0.5', 'eval_soft_accuracy': '0.3373', 'eval_

{'colour': {'type': 'choice',
  'choice': 'blue',
  'probabilities': {'red': 0.3323, 'green': 0.3332, 'blue': 0.3345},
  'confidence': 0.0,
  'answer_confidence': 0.3345}}

In [ ]:
test_eq([tiles_for(s) for s in (256, 512, 1024, 1536, None)], [1, 1, 5, 10, 17])
many = decision_learner(cdata, FIX, image_side=128, preset="cpu", cache_dir=tempfile.mkdtemp(), calibrate_after_fit=False)   # 2 × 2 tiles + the global view
test_eq((many.args.per_device_train_batch_size, many.args.per_device_eval_batch_size), (8, 16 // 5))    # cached: only the encoding batch shrinks
lora = decision_learner(cdata, FIX, train="lora", image_side=128, preset="cpu", calibrate_after_fit=False)
test_eq((lora.args.per_device_train_batch_size, lora.args.per_device_eval_batch_size), (8 // 5 or 1, 16 // 5))   # no cache: both do
del many, lora

At 128 pixels, a fixture image is cut into 2 × 2 tiles, and `pixel_tiles` puts them back together: an image whose quarters are four colours comes back with its quarters in place, beside the global view:

In [ ]:
quarters = PILImage.new("RGB", (128, 128))
for c, xy in zip(list(COLOURS.values()) + [(240, 240, 240)], [(0, 0), (64, 0), (0, 64), (64, 64)]): quarters.paste(c, xy + (xy[0] + 64, xy[1] + 64))
p4 = AutoProcessor.from_pretrained(FIX); idefics3_prepare(p4, 128)                  # a processor of its own, tiling at 128 pixels
four = RowBuilder(p4, spec.template, [Image(side=128)])
qrow = four.build_record({"state": {"text": "", "image": quarters}, "questions": cq})[0]
qtiles, qvals = pixel_tiles(four.processor, qrow, np.ones((1,) + tuple(qrow.extras["pixel_values"].shape)))
test_eq([t.shape for t in qtiles], [(128, 128, 3), (64, 64, 3)])
test_eq([v.shape for v in qvals], [(1, 128, 128), (1, 64, 64)])
test_close(qtiles[0][32, 32], np.array(COLOURS["red"]) / 255, eps=0.02); test_close(qtiles[0][96, 96], np.full(3, 240 / 255), eps=0.02)

::: {.callout-note title="Finding: tiles shrank a cached head's training batch"}
The application shrinks the batches for a tiling processor, since each row then carries several images through the vision tower. It shrank the training batch too, even when the head trains from cached anchors and the vision tower runs only to fill the cache. So at 1,024 pixels, five images a row, a cached head trained on batches of one row instead of four: a different recipe from the one it gets at 512. Now, with cached anchors, only the batches that encode images shrink. Without a cache (LoRA, full fine-tuning) the vision tower runs in every training step, and the training batch shrinks as before. The [ModernVBERT tutorial](tutorials/modernvbert_images.ipynb#one-more-setting-the-image-size)'s image-size sweep ran before the fix, so there the default head trained on batches of one row at 1,024 and 1,536 pixels. Its linear head, fitted in one full batch, is unaffected.
:::

The LoRA regime puts adapters on the text tower's `Wqkv` and `Wo` alone, and trains them with the head:

In [ ]:
torch.manual_seed(0)
ll = decision_learner(cdata, FIX, train="lora", image_side=64, preset="cpu", loss="soft_ce", calibrate_after_fit=False)
trainable = [n for n, q in ll.model.encoder.named_parameters() if q.requires_grad]
test_eq((ll.cache, len(trainable) > 0, all("text_model" in n and "lora_" in n for n in trainable)), (None, True, True))
ll.fit(1, lr=1e-3)

{'eval_loss': '1.097', 'eval_accuracy': '0.3333', 'eval_soft_accuracy': '0.3338', 'eval_brier': '0.6658', 'eval_kl': '1.097', 'eval_tv': '0.6662', 'eval_ece': '0.001924', 'eval_score_mae': 'nan', 'eval_within_one': 'nan', 'eval_n': 6, 'eval_accuracy_choice': '0.3333', 'eval_accuracy_score': 'nan', 'eval_accuracy_noul': 'nan', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0074', 'eval_samples_per_second': '805.9', 'eval_steps_per_second': '134.3', 'epoch': '1'}
{'train_runtime': '0.0286', 'train_samples_per_second': '489.6', 'train_steps_per_second': '69.94', 'train_loss': '1.098', 'epoch': '1'}


{'epoch': 1.0,
 'loss': 1.0973469018936157,
 'accuracy': 0.3333333333333333,
 'soft_accuracy': 0.33376070815662184,
 'brier': 0.6658196388733407,
 'kl': 1.0973469185878548,
 'tv': 0.6662392918433783,
 'ece': 0.001924100426870412,
 'score_mae': nan,
 'within_one': nan,
 'n': 6,
 'accuracy_choice': 0.3333333333333333,
 'accuracy_score': nan,
 'accuracy_noul': nan,
 'n_unanswerable': 0,
 'runtime': 0.0074,
 'samples_per_second': 805.874,
 'steps_per_second': 134.312}

## 7. Two small image tasks on the real checkpoint

The [ModernVBERT tutorial](tutorials/modernvbert_images.ipynb) runs the multimodal application with `ModernVBERT/modernvbert` as its encoder, with every output shown. It uses the NeoMME tutorial's records and recipes, and the whole run takes 23 minutes on an M3 Pro. Held-out accuracy:

| | default head | linear head | LoRA | probe on the image vectors |
| --- | --- | --- | --- | --- |
| digits (a 10-way choice, 100 test images) | 0.84 | 0.77 | 0.84 | 0.92 |
| hot dog (a noul, 80 test photos) | 0.94 | 0.93 | 0.96 | 0.98 |

With the same recipes NeoMME scored 0.31, 0.41 and 0.33 on the digits, and 0.74, 0.70 and 0.73 on the photos (see the [comparison](tutorials/neomme_vs_modernvbert.ipynb)). Standardising the anchors still matters: on the digits the head reaches 0.67 without it, since ModernVBERT's channel 256 carries 58% of every anchor's length. More tiles helped neither task. One 512-pixel tile, 102 tokens a row, did as well as 5 or 10.

## 8. What to watch

- **The processor enlarges images.** The checkpoint's processor resizes every image to 2,048 pixels on its long side before tiling it, so a thumbnail left at that size costs 1,127 positions. The multimodal application's default is one tile, 512 pixels. A large page may need 1,024 or more.
- **Below 512 pixels, `image_side` changes the detail, not the cost.** Every tile is 512 pixels when it reaches SigLIP, so a smaller side only blurs the image first.
- **The load report.** `ModernVBERT/modernvbert` holds its masked-LM head and a projection head beside the encoder, and transformers reports them as unused when it loads the encoder. The weights are the encoder's all the same.
- **Tiles multiply the vision tower's batch.** A row at `image_side` 1,024 is 5 tiles and at 1,536 it is 10, each 1,024 SigLIP patches. The application divides the preset's batch sizes by the tiles per image (the finding in [section 6](#a-head-only-run)). A batch size given by hand is kept as it is, so size it the same way.
- **Batches are not bit-for-bit.** On MPS a row's anchors differ from those of the same row alone by up to 3 × 10⁻⁵, the reordering noise of float32 kernels. The feature cache and the `Decider` see the same numbers only when they batch the same way.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()